In [ ]:
%pip install agentpy
%pip install seaborn

In [ ]:
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from IPython.display import HTML
import agentpy as ap
from matplotlib.animation import PillowWriter

In [580]:
def fillGrid(rows, cols, arm_len=11, half_width=7):
    grid = np.ones((rows, cols, 3), dtype=np.uint8) * 255  # white background

    black  = [0, 0, 0]
    green  = [0, 255, 0]
    red    = [255, 0, 0]
    blue   = [0, 0, 255]
    orange = [255, 165, 0]

    mid_r = rows // 2
    mid_c = cols // 2
    border_rows = [mid_r - half_width, mid_r + half_width]
    border_cols = [mid_c - half_width, mid_c + half_width]

    # Left & Right
    for r in border_rows:
        grid[r, 0:arm_len] = black
        grid[r, cols - arm_len:cols] = black

    grid[0:arm_len, mid_c] = black
    grid[rows-arm_len:rows, mid_c] = black

    # Top & Bottom
    for c in border_cols:
        grid[0:arm_len, c] = black
        grid[rows - arm_len:rows, c] = black

    grid[mid_r, 0:arm_len] = black
    grid[mid_r, cols-arm_len:cols] = black

    # Example traffic lights
    grid[mid_c-half_width-1, arm_len-2] = green  # Top Left
    grid[mid_c+half_width+1, arm_len-2] = green  # Bottom Left
    grid[arm_len-2, mid_r+half_width+1] = green  # Top Right
    grid[rows-arm_len+1, mid_r+half_width+1] = green  # Bottom Right


    # #Blue: Origin, Red: Destination, Orange: Stop Point for traffic Light.
    # # Top Sur
    # grid[0, mid_c - half_width//2] = blue 
    # grid[arm_len, mid_c - half_width//2] = orange 
    # grid[0, mid_c + half_width//2] = red

    # # Bottom Norte
    # grid[rows-1, mid_c - half_width//2] = red
    # grid[rows-arm_len-1, mid_c + half_width//2] = orange
    # grid[rows-1, mid_c + half_width//2] = blue

    # # Left Este
    # grid[mid_r - half_width//2, 0] = red
    # grid[mid_r + half_width//2, arm_len] = orange
    # grid[mid_r + half_width//2, 0] = blue

    # # Right Oeste
    # grid[mid_r - half_width//2, cols-1] = blue
    # grid[mid_r - half_width//2, cols-arm_len-1] = orange
    # grid[mid_r + half_width//2, cols-1] = red

    return grid


In [581]:
def fillGridData(rows, cols, arm_len=11, half_width=7):
    collisionLocations = []
    lightLocations = {}
    locationToCoords = {}
    
    # Center lines that define the two 'borders' for each approach
    mid_r = rows // 2
    mid_c = cols // 2
    border_rows = [mid_r - half_width, mid_r + half_width]  # for left/right sides
    border_cols = [mid_c - half_width, mid_c + half_width]  # for top/bottom sides

    # Left & Right sides (vertical borders → horizontal segments going inward)
    for r in border_rows:
        collisionLocations.append((r,i) for i in range(arm_len))
        collisionLocations.append((r,i) for i in range(cols - arm_len,cols))
        
        
    collisionLocations.append((i,mid_c) for i in range(arm_len))
    collisionLocations.append((i,mid_c) for i in range(rows-arm_len,rows))
    # Top & Bottom sides (horizontal borders → vertical segments going inward)
    for c in border_cols:
        collisionLocations.append((i,c) for i in range(arm_len))
        collisionLocations.append((i,c) for i in range(rows - arm_len,rows))

    collisionLocations.append((mid_r,i) for i in range(arm_len))
    collisionLocations.append((mid_r,i) for i in range(cols-arm_len,cols))

    lightLocations = {
        1: (rows-arm_len+1, mid_r+half_width+1), #Junco Norte
        3:(mid_c+half_width+1, arm_len-2), #Roel Este 
        5:(mid_c-half_width-1, arm_len-2), #Junco Sur
        7:(arm_len-2, mid_r+half_width+1) #Roel Oeste
    }

    locationToCoords = { #Spawn locations
        1: (rows-1,mid_c + half_width//2),#Junco Norte
        2: (rows-1,mid_c - half_width//2),#Junco Norte
        3: (mid_r + half_width//2, 0),#Roel Este
        4: (mid_r - half_width//2, 0), #Roel Este
        5: (0,mid_c - half_width//2),#Junco Sur
        6: (0,mid_c + half_width//2),#Junco Sur
        7: (mid_r - half_width//2, cols-1), #Roel Oeste
        8: (mid_r + half_width//2, cols-1)#Roel Oeste
    }

    lightCheckpoints = {
        1: (rows-arm_len-1, mid_c + half_width//2), #Junco Norte
        3: (mid_r + half_width//2, arm_len),#Roel Este
        5: (arm_len, mid_c - half_width//2), #Junco Sur
        7: (mid_r - half_width//2, cols-arm_len-1), #Roel Oeste
    }

    return collisionLocations, lightLocations, locationToCoords,lightCheckpoints

In [582]:
import numpy as np
import matplotlib.pyplot as plt

def my_plot(model, ax, arm_len=11, half_width=7):
   
    rows, cols = model.environment.shape  # expected (35, 35)

    colorDict = {
        'green' : [0,255,0],
        'red' : [255,0,0],
        'yellow' : [255, 255, 0]  
    }
    
    grid = fillGrid(rows,cols)

    for agent, pos in model.environment.positions.items():
        if agent.__class__.__name__ == "TrafficLight":
            grid[pos] = colorDict[agent.get_color()] 
        elif agent.__class__.__name__ == "Vehicle":
            grid[pos] = [106,13,173]


    ax.imshow(grid)
    ax.set_xticks([])
    ax.set_yticks([])

# class DummyEnv:
#     shape = (35, 35)

# class DummyModel:
#     environment = DummyEnv()

# fig, ax = plt.subplots(figsize=(5, 5))
# my_plot(DummyModel(), ax) 
# plt.show()

In [583]:
#Car Probabilities

morning_probs = {
    1: 0.9, #Junco Norte 
    3: 0.9, #Roel Este 
    5: 0.9, #Junco Sur 
    7: 0.9, #Roel Oeste 
}

afternoon_probs = {
    1: 0.9, #Junco Norte 
    3: 0.9, #Roel Este 
    5: 0.9, #Junco Sur 
    7: 0.9, #Roel Oeste 
}

night_probs = {
    1: 0.9, #Junco Norte 
    3: 0.9, #Roel Este 
    5: 0.9, #Junco Sur 
    7: 0.9, #Roel Oeste 
}


In [584]:
#Car Amounts

morning_cars = {
    (1, 4): 3, #Junco Norte Izquierda
    (1, 6): 3, #Junco Norte Adelante
    (1, 8): 1, #Junco Norte Derecha
    (3, 6): 1, #Roel Este Izquierda
    (3, 8): 1, #Roel Este Adelante
    (3, 2): 3, #Roel Este Derecha
    (5, 8): 1, #Junco Sur Izquierda
    (5, 2): 1, #Junco Sur Adelante
    (5, 4): 1, #Junco Sur Derecha
    (7, 2): 3, #Roel Oeste Izquierda
    (7, 4): 7, #Roel Oeste Adelante
    (7, 6): 5 #Roel Oeste Derecha
}

afternoon_cars = {
    (1, 4): 3, #Junco Norte Izquierda
    (1, 6): 3, #Junco Norte Adelante
    (1, 8): 1, #Junco Norte Derecha
    (3, 6): 3, #Roel Este Izquierda
    (3, 8): 3, #Roel Este Adelante
    (3, 2): 1, #Roel Este Derecha
    (5, 8): 1, #Junco Sur Izquierda
    (5, 2): 3, #Junco Sur Adelante
    (5, 4): 1, #Junco Sur Derecha
    (7, 2): 3, #Roel Oeste Izquierda
    (7, 4): 5, #Roel Oeste Adelante
    (7, 6): 5 #Roel Oeste Derecha
}

night_cars = {
    (1, 4): 3, #Junco Norte Izquierda
    (1, 6): 3, #Junco Norte Adelante
    (1, 8): 1, #Junco Norte Derecha
    (3, 6): 1, #Roel Este Izquierda
    (3, 8): 1, #Roel Este Adelante
    (3, 2): 3, #Roel Este Derecha
    (5, 8): 1, #Junco Sur Izquierda
    (5, 2): 3, #Junco Sur Adelante
    (5, 4): 1, #Junco Sur Derecha
    (7, 2): 5, #Roel Oeste Izquierda
    (7, 4): 5, #Roel Oeste Adelante
    (7, 6): 3 #Roel Oeste Derecha
}



In [585]:
# Locations
locations = {
    1: {"name": "Junco de la Vega Norte Origen", "location": (0, 0), "type": "origin"},
    2: {"name": "Junco de la Vega Norte Destino", "location": (0, 0), "type": "destination"},
    3: {"name": "Garcia Roel Este Origen",       "location": (0, 0), "type": "origin"},
    4: {"name": "Garcia Roel Este Destino",      "location": (0, 0), "type": "destination"},
    5: {"name": "Junco de la Vega Sur Origen",   "location": (0, 0), "type": "origin"},
    6: {"name": "Junco de la Vega Sur Destino",  "location": (0, 0), "type": "destination"},
    7: {"name": "Garcia Roel Oeste Origen",      "location": (0, 0), "type": "origin"},
    8: {"name": "Garcia Roel Oeste Destino",     "location": (0, 0), "type": "destination"}
}

#Traffic Light Phases
phaseDuration = {
    1: {"origin" : 1, "isVehicular": True ,"time": 20},
    3: {"origin" : 2, "isVehicular": True, "time": 19},
    5: {"origin" : 3, "isVehicular": True, "time": 19},
    7: {"origin" : 4, "isVehicular": True, "time": 22},
    9: {"origin" : -1, "isVehicular": False, "time": 10}, #All Traffic Lights are on red for pedestrians
}

In [586]:
def generate_flows(time: str, vehicle_pattern, car_probs):
    
    flows = {}
    for origin in range(1, 9, 2):  # origins 1-4
        for destination in range(2, 9, 2):
            if origin + 1 == destination:
                continue  # skip same-node trips
                
            vehicles = vehicle_pattern[(origin, destination)]
            car_prob = car_probs[origin]

            flows[(origin, destination)] = {
                "time": time,
                "vehicles": vehicles,
                "carProbability": car_prob
            }
    return flows




In [587]:
# Generate datasets
morningAmount   = generate_flows("morning", morning_cars, morning_probs)
afternoonAmount = generate_flows("afternoon", afternoon_cars, afternoon_probs)  # default pattern
nightAmount     = generate_flows("night", night_cars, night_probs)      # default pattern

# Group everything
carAmounts = {
    "morning": morningAmount, #2-10
    "afternoon": afternoonAmount, #10-18
    "night": nightAmount #18-2
}

carAmounts

{'morning': {(1, 4): {'time': 'morning', 'vehicles': 3, 'carProbability': 0.9},
  (1, 6): {'time': 'morning', 'vehicles': 3, 'carProbability': 0.9},
  (1, 8): {'time': 'morning', 'vehicles': 1, 'carProbability': 0.9},
  (3, 2): {'time': 'morning', 'vehicles': 3, 'carProbability': 0.9},
  (3, 6): {'time': 'morning', 'vehicles': 1, 'carProbability': 0.9},
  (3, 8): {'time': 'morning', 'vehicles': 1, 'carProbability': 0.9},
  (5, 2): {'time': 'morning', 'vehicles': 1, 'carProbability': 0.9},
  (5, 4): {'time': 'morning', 'vehicles': 1, 'carProbability': 0.9},
  (5, 8): {'time': 'morning', 'vehicles': 1, 'carProbability': 0.9},
  (7, 2): {'time': 'morning', 'vehicles': 3, 'carProbability': 0.9},
  (7, 4): {'time': 'morning', 'vehicles': 7, 'carProbability': 0.9},
  (7, 6): {'time': 'morning', 'vehicles': 5, 'carProbability': 0.9}},
 'afternoon': {(1, 4): {'time': 'afternoon',
   'vehicles': 3,
   'carProbability': 0.9},
  (1, 6): {'time': 'afternoon', 'vehicles': 3, 'carProbability': 0.9},

In [588]:
class TrafficControler(ap.Agent):

    def setup(self, env, trafficLightList, phaseDuration):
        self.env = env
        self.trafficLightList = trafficLightList
        self.phaseDuration = phaseDuration
        self.time = 0
        self.eventIdx = 0
        self.currentPhase = 1
        self.pedestrianPhase = 9
        self.currentSeconds = 0
        

    def execute(self):
        if(self.currentSeconds > self.phaseDuration[self.currentPhase]["time"]):
            self.currentSeconds = 0
            if(self.currentPhase != self.pedestrianPhase):
                self.trafficLightList[self.currentPhase].set_color('red')
            self.currentPhase = 1 if self.currentPhase == self.pedestrianPhase else self.currentPhase + 2
            if(self.currentPhase != self.pedestrianPhase):
                self.trafficLightList[self.currentPhase].set_color('green')
        
        if(self.currentSeconds >= self.phaseDuration[self.currentPhase]["time"] - 5):
            if(self.currentPhase != self.pedestrianPhase):
                self.trafficLightList[self.currentPhase].set_color('yellow')

        self.currentSeconds += 1

In [589]:
class TrafficLight(ap.Agent):

    def setup(self, env, location):
        self.env = env
        self.color = 'red' #green,yellow,red
        self.location = location
        
    def get_color(self):
        return self.color
    
    def set_color(self, color):
        self.color = color
    
    def get_location(self):
        return self.location
        

    def execute(self):
        1+1
        #print(f'Execute trafficlight at: {self.location} on color: {self.color}')

In [590]:
class Vehicle(ap.Agent):

    #def setup(self, env, isCar, inRightLane, direction, destination):
    def setup(self, env, trafficLight, startPos, lightCheckpoint, destination):
        self.env = env
        self.trafficLight = trafficLight
        self.lightCheckpoint = lightCheckpoint
        self.destination = destination
        rowPos,colPos = startPos
        self.isMoving = False
        self.passedLight = False
        self.direction = ()
        self.rows, self.cols = self.env.shape
        self.toRemove = False
        self.directionChange = False
        self.movingUnits = 1
        dest_x, dest_y = self.destination
        if(rowPos == 0): #Top
            self.direction = (self.movingUnits,0)
        elif(colPos == 0): #Left
            self.direction = (0,self.movingUnits)
        elif(rowPos == 34): #Bottom NEEDS CHANGE
            self.direction = (-self.movingUnits,0)
        else:
            self.direction = (0,-self.movingUnits)

        if(dest_x == rowPos or dest_y ==colPos):
            self.directionChange = True
            

    def execute(self):
        x,y = np.array(self.direction) + self.env.positions[self]
        if(self.env.positions[self] == self.lightCheckpoint):
            self.passedLight = True
        if not (0 <= x < self.rows and 0 <= y < self.cols):
            self.toRemove = True
        elif(self.passedLight or self.trafficLight.get_color() != 'red'):
            self.env.move_by(self, self.direction)
        elif (not self.passedLight and (x,y) != self.lightCheckpoint and (x,y) not in self.env.positions.values()):
            self.env.move_by(self, self.direction)

        if not self.directionChange:
            dest_x, dest_y = self.destination
            # Coming from vertical (moving up or down), check y alignment
            if (self.direction in [(self.movingUnits,0), (-self.movingUnits,0)] and x == dest_x):
                # turn horizontal
                if dest_y > y:   # destination to the right
                    self.direction = (0, self.movingUnits)
                else:                # destination to the left
                    self.direction = (0, -self.movingUnits)
                self.directionChange = True

            # Coming from horizontal (moving left or right), check x alignment
            elif (self.direction in [(0,self.movingUnits), (0,-self.movingUnits)] and y == dest_y):
                # turn vertical
                if (dest_x > x):   # destination below
                    self.direction = (self.movingUnits, 0)
                else:                # destination above
                    self.direction = (-self.movingUnits, 0)
                self.directionChange = True

        

In [591]:
class TrafficModel(ap.Model):

    def setup(self):
        self.morningAmounts = self.p.carAmounts["morning"]
        self.afternoonAmounts = self.p.carAmounts["afternoon"]
        self.nightAmounts = self.p.carAmounts["night"]
        self.locations = self.p.locations
        self.phaseDuration = self.p.phaseDuration
        self.lightLocations = self.p.lightLocations
        self.collisionLocations = self.p.collisionLocations
        self.locationToCoords = self.p.locationToCoords
        self.lightCheckpoints = self.p.lightCheckpoints
        self.currentHour = 0
        self.environment = ap.Grid(self, (35, 35))
        self.timeInSeconds = 0
        self.cycleDuration = 90
        self.trafficLightDict = {}
        self.agentsQueue = {i:[] for i in self.lightLocations.keys()}
        
        for i in self.lightLocations.keys():
            currentTrafficLignt = TrafficLight(self,self.environment, self.lightLocations[i])
            self.trafficLightDict[i] = currentTrafficLignt
            self.environment.add_agents([currentTrafficLignt], positions=[self.lightLocations[i]])


        self.trafficControlerAgent = TrafficControler(self, self.environment, self.trafficLightDict, self.phaseDuration)
        self.environment.add_agents([self.trafficControlerAgent], positions=[(0,0)])

        self.trafficLightDict[1].set_color('green')

    def step(self):

        #Spawn cars if its the light is on red
        for lightKey in self.trafficLightDict.keys():
            if(self.trafficLightDict[lightKey].get_color() == 'red' and self.agentsQueue[lightKey]):
                currentAgent, position = self.agentsQueue[lightKey][-1]
                self.environment.add_agents(currentAgent, positions=position)
                self.agentsQueue[lightKey].pop()
        
        if(self.timeInSeconds == 0): #Fill the queues at the pedestrian phase.
            for lightKey in self.trafficLightDict.keys():
                if(not self.agentsQueue[lightKey]):
                    for origin,destination in self.morningAmounts.keys():
                        if(origin == lightKey):
                            self.agentsQueue[lightKey].append(([Vehicle(self,self.environment, self.trafficLightDict[lightKey],self.locationToCoords[lightKey],
                                self.lightCheckpoints[lightKey],self.locationToCoords[destination])],[self.locationToCoords[lightKey]]))
        
        
        to_remove = [agent for agent in self.environment.agents if agent.__class__.__name__ == "Vehicle" and agent.toRemove]
        self.environment.remove_agents(to_remove)
        self.timeInSeconds = (self.timeInSeconds + 1) % (self.cycleDuration + 1)
        self.environment.agents.execute()

        

In [ ]:
fig, ax = plt.subplots()
collisionLocations, lightLocations, locationToCoords, lightCheckpoints = fillGridData(35,35)
parameters = {'steps': 90*5,'carAmounts': carAmounts, 'locations':locations, 'phaseDuration': phaseDuration, 
    'lightLocations':lightLocations, 'collisionLocations' : collisionLocations, 'locationToCoords': locationToCoords,
    'lightCheckpoints' : lightCheckpoints } 
trafficModel = TrafficModel(parameters)
animation = ap.animate(trafficModel, fig, ax, my_plot)
writergif = PillowWriter(fps=10) # Set desired frames per second
animation.save('my_animation.gif', writer=writergif)
HTML(animation.to_jshtml())
